# CRIVO — Fase 7a: crescer o v7c para ~290M e ler (A100)

O CRIVO **cresce sem ser trocado**, como na fase 6a (17M → 46M): o 46M do chat (v7c, `e4067e3b…`, o que conversa hoje)
dobra de largura (384 → 768, 8 → 16 cabeças de 48) e ganha 16 camadas (24 → 40), ficando com **~290M de parâmetros**
(`scripts/crescer_46m.py`). Cada número do modelo vira dois iguais e cada camada nova começa neutra: antes de treinar,
o crescido diz **o mesmo** que o v7c (conferido aqui na Célula 3: a diferença de logit e a palavra mais provável).

**O que ele lê:** o mesmo texto da fase 6a (FineWeb-2 português, ODC-By; corpus próprio: Wikipédia pt, OASST2, acervo),
mais ~1,5 bilhão de tokens novos do FineWeb-2 (só na primeira vez; ficam no Drive). Ao todo, uns **3 bilhões de tokens**,
45 mil passos de 64 × 1024 tokens. Nada de modelo de terceiros: é texto escrito por pessoas.

**Depois desta fase** ele ainda não conversa tão bem quanto o v7c (ler texto da web desgasta o formato de conversa): a
**fase 7b** repete a receita do v7c (mesmos dados, mesma régua própria nas duas rodadas, contra o v7c) a partir desta base.
Só entra no chat se passar na régua.

**Tempo:** ~1,4 s por passo numa A100 de 40 GB → umas **17–18 horas**, em duas sessões. "Executar tudo" de novo continua do
último checkpoint (salvo a cada 2 mil passos no Drive, ~3,5 GB cada; os substituídos vão para a lixeira do Drive).
Na primeira vez, baixar e tokenizar o texto novo leva mais 1–2 horas.

> **A100 → Executar tudo.** Saída: `CRIVO/transformer-crivo-300m/`; resumo pequeno: `resumo_fase7a.json`.

In [ ]:
# Célula 0 — GPU, Drive e o 46M do chat (v7c)
import os, sys, json, shutil, subprocess, hashlib, time, math, random, glob
from pathlib import Path
from datetime import datetime
import numpy as np
import torch

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → A100.")
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| PyTorch", torch.__version__, flush=True)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

BASE_SHA = "e4067e3b4c9705fde0a2063cc7a0da958ebad196ca193a672f75a1a10cbfda0a"   # 46M v7c, o do chat (conversa_v3d.PESOS_SHA256_46M)
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-crivo-46m-conversa-v7c").is_dir()), MEU / "CRIVO")
BASE = next((Path(p) for p in sorted(glob.glob(str(CRIVO / "transformer-crivo-46m-conversa-v7c" / "**" / "pesos.pt"), recursive=True))
             if sha256(p) == BASE_SHA), None)
if BASE is None:
    raise SystemExit(f"ERRO: não achei o v7c (sha e4067e3b…) em {CRIVO}/transformer-crivo-46m-conversa-v7c/. "
                     "Rode na conta onde o v7c treinou ou numa conta com atalho para a pasta CRIVO.")
SAIDA = CRIVO / "transformer-crivo-300m"
SAIDA.mkdir(parents=True, exist_ok=True)
CORPUS_DRIVE = CRIVO / "corpus-fineweb2-pt"
CORPUS_DRIVE.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/fase7a")
LOCAL.mkdir(parents=True, exist_ok=True)
dispositivo = torch.device("cuda")
SEED = 20261020
print("CRIVO:", CRIVO, "| v7c:", BASE, flush=True)


In [ ]:
# Célula 1 — código do CRIVO e bibliotecas (tokenizers, leitura de parquet do Hugging Face)
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1", "huggingface_hub>=0.24", "pyarrow>=14"], check=True)
for p in (str(REPO), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import crescer_46m
from tokenizers import Tokenizer
TOK = Tokenizer.from_file(str(BASE.parent / "tokenizer.json"))
DOC, FIM = TOK.token_to_id("<documento>"), TOK.token_to_id("<fim>")
print("commit:", COMMIT, flush=True)


In [ ]:
# Célula 2 — o texto: corpus próprio + FineWeb-2 português, em tokens do CRIVO (uma vez; depois vem do Drive)
META_WEB = 4_000_000_000        # tokens de web: os 2,5 bi da fase 6a e ~1,5 bi novos
TOKENS_POR_PARTE = 250_000_000  # cada parte no Drive tem ~500 MB
VAL_WEB = 3_000_000             # tokens separados para medir, nunca treinados
for n in ("treino.bin", "validacao.bin"):
    if not (LOCAL / n).is_file():
        shutil.copyfile(CRIVO / "dados-17m" / n, LOCAL / n)
PROPRIO = np.memmap(LOCAL / "treino.bin", dtype=np.uint16, mode="r")
PROPRIO_VAL = np.memmap(LOCAL / "validacao.bin", dtype=np.uint16, mode="r")
MANIFESTO = CORPUS_DRIVE / "manifesto.json"
man = json.loads(MANIFESTO.read_text()) if MANIFESTO.is_file() else {"partes": [], "arquivos_usados": [], "val": None,
                                                                     "fonte": "HuggingFaceFW/fineweb-2 por_Latn (ODC-By 1.0)"}

def baixar_web():
    """Gera textos do FineWeb-2 português, arquivo por arquivo, a partir de onde parou."""
    from huggingface_hub import list_repo_files, hf_hub_download
    import pyarrow.parquet as pq
    arquivos = sorted(f for f in list_repo_files("HuggingFaceFW/fineweb-2", repo_type="dataset")
                      if f.startswith("data/por_Latn/train/") and f.endswith(".parquet"))
    for nome in arquivos:
        if nome in man["arquivos_usados"]:
            continue
        caminho = hf_hub_download("HuggingFaceFW/fineweb-2", nome, repo_type="dataset", local_dir=str(LOCAL / "hf"))
        for lote in pq.ParquetFile(caminho).iter_batches(batch_size=4000, columns=["text"]):
            yield nome, [t for t in lote.column(0).to_pylist() if t and len(t) >= 200]
        os.remove(caminho)
        yield nome, None   # fim do arquivo

def tokenizar(textos):
    return [np.asarray([DOC] + e.ids + [FIM], dtype=np.uint16) for e in TOK.encode_batch(textos, add_special_tokens=False)]

total_web = sum(p["tokens"] for p in man["partes"])
if total_web < META_WEB or man["val"] is None:
    t0, buffer, n_buf = time.time(), [], 0
    for nome, textos in baixar_web():
        if textos is None:
            man["arquivos_usados"].append(nome)
            MANIFESTO.write_text(json.dumps(man, indent=1))
            continue
        for ids in tokenizar(textos):
            buffer.append(ids); n_buf += len(ids)
        alvo = VAL_WEB if man["val"] is None else TOKENS_POR_PARTE
        if n_buf >= alvo:
            dados = np.concatenate(buffer); buffer, n_buf = [], 0
            if man["val"] is None:
                (CORPUS_DRIVE / "val.bin").write_bytes(dados.tobytes())
                man["val"] = {"arquivo": "val.bin", "tokens": int(len(dados)), "sha256": sha256(CORPUS_DRIVE / "val.bin")}
            else:
                nome_parte = "parte_%03d.bin" % len(man["partes"])
                (CORPUS_DRIVE / nome_parte).write_bytes(dados.tobytes())
                man["partes"].append({"arquivo": nome_parte, "tokens": int(len(dados)), "sha256": sha256(CORPUS_DRIVE / nome_parte)})
                total_web += len(dados)
                print(f"  {nome_parte}: {len(dados)/1e6:.0f}M tokens | web total {total_web/1e9:.2f} bi ({time.time()-t0:.0f}s)", flush=True)
            MANIFESTO.write_text(json.dumps(man, indent=1))
            if total_web >= META_WEB:
                break
print("web:", f"{total_web/1e9:.2f} bi de tokens em {len(man['partes'])} partes | próprio:", f"{len(PROPRIO)/1e9:.2f} bi", flush=True)
for p in man["partes"] + [man["val"]]:
    local = LOCAL / p["arquivo"]
    if not local.is_file() or local.stat().st_size != 2 * p["tokens"]:
        shutil.copyfile(CORPUS_DRIVE / p["arquivo"], local)
WEB = [np.memmap(LOCAL / p["arquivo"], dtype=np.uint16, mode="r") for p in man["partes"]]
WEB_PESOS = np.array([len(w) for w in WEB], dtype=np.float64); WEB_PESOS /= WEB_PESOS.sum()
WEB_VAL = np.memmap(LOCAL / man["val"]["arquivo"], dtype=np.uint16, mode="r")


In [ ]:
# Célula 3 — o CRIVO crescido (768 × 40, ~290M), ou o último checkpoint
CTX = 1024
CKPT = SAIDA / "checkpoint.pt"
salvo = torch.load(BASE, map_location="cpu", weights_only=True)
estado, CFG = crescer_46m.crescer_estado(salvo["modelo"], dict(salvo["config"], dropout=0.0), camadas=40, ruido=1e-4)
CFG["dropout"] = 0.0
modelo = lp.LinguagemProfunda(lp.Configuracao(**CFG)).to(dispositivo)
modelo.load_state_dict(estado); del estado
original = lp.LinguagemProfunda(lp.Configuracao(**dict(salvo["config"], dropout=0.0))).to(dispositivo).eval()
original.load_state_dict(salvo["modelo"])
with torch.no_grad():
    x = torch.from_numpy(PROPRIO_VAL[:512].astype(np.int64))[None].to(dispositivo)
    a, b = original(x)[0], modelo.eval()(x)[0]
    dif, top1 = float((a - b).abs().max()), float((a.argmax(-1) == b.argmax(-1)).float().mean())
print(f"crescido: {CFG['dimensao']} × {CFG['camadas']} camadas, {CFG['cabecas']} cabeças, "
      f"{sum(p.numel() for p in modelo.parameters())/1e6:.1f}M parâmetros | contra o v7c: maior diferença de logit {dif:.2e}, "
      f"mesma palavra mais provável em {100*top1:.1f}%", flush=True)
if top1 < 0.98:
    raise SystemExit("ERRO: o crescimento mudou o que o modelo diz (esperado ≥ 98% iguais, com o ruído pequeno). Avise o assistente.")
del original, salvo
PASSO0, HIST = 0, []
if CKPT.is_file():
    c = torch.load(CKPT, map_location="cpu", weights_only=False)
    modelo.load_state_dict(c["modelo"]); PASSO0, HIST = c["passo"], c["historico"]
    print("continuando do checkpoint, passo", PASSO0, flush=True)


In [ ]:
# Célula 4 — ler: 70% web, 30% corpus próprio; lote de 64 em pedaços; medidas e checkpoint a cada 2 mil passos
PASSOS = 45_000                 # ~3 bilhões de tokens
LOTE, FRAC_WEB = 64, 0.7
LR, LR_MIN, AQUECIMENTO = 1.5e-4, 1.5e-5, 2000   # mais baixa que a da 6a: o modelo já sabe; as partes novas aprendem
SALVAR_A_CADA, MEDIR_A_CADA, MAX_HORAS = 2000, 2000, 11.5   # cada checkpoint substituído vai para a lixeira do Drive (~3,5 GB)
MEMORIA_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
PEDACOS = 2 if MEMORIA_GB > 60 else 4    # A100 80 GB: 2 pedaços de 32; 40 GB: 4 de 16
print(f"GPU com {MEMORIA_GB:.0f} GB: lote de {LOTE} em {PEDACOS} pedaços", flush=True)

params = list(modelo.parameters())
otim = torch.optim.AdamW([{"params": [p for p in params if p.dim() >= 2], "weight_decay": 0.1},
                          {"params": [p for p in params if p.dim() < 2], "weight_decay": 0.0}], lr=LR, betas=(0.9, 0.95), fused=True)
if CKPT.is_file():
    otim.load_state_dict(c["otim"]); del c
escala = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
try:
    rodar = torch.compile(modelo)
except Exception as erro:
    print("sem torch.compile:", erro, flush=True); rodar = modelo
taxa = lambda p: LR * (p + 1) / AQUECIMENTO if p < AQUECIMENTO else LR_MIN + (LR - LR_MIN) * 0.5 * (1 + math.cos(math.pi * min(1.0, (p - AQUECIMENTO) / max(1, PASSOS - AQUECIMENTO))))
rng = np.random.default_rng(SEED + PASSO0)

def janelas(fonte, n, ctx):
    ini = rng.integers(0, len(fonte) - ctx - 1, size=n)
    return np.stack([fonte[i:i + ctx + 1] for i in ini]).astype(np.int64)

def lote():
    n_web = int(round(LOTE * FRAC_WEB))
    partes = [janelas(WEB[i], int(k), CTX) for i, k in enumerate(np.bincount(rng.choice(len(WEB), n_web, p=WEB_PESOS), minlength=len(WEB))) if k]
    partes.append(janelas(PROPRIO, LOTE - n_web, CTX))
    b = torch.from_numpy(np.concatenate(partes)).to(dispositivo, non_blocking=True)
    return b[:, :-1], b[:, 1:]

@torch.no_grad()
def perda(fonte, ctx, lotes=30, tam=8):
    modelo.eval(); g = np.random.default_rng(SEED); r = []
    for _ in range(lotes):
        ini = g.integers(0, len(fonte) - ctx - 1, size=tam)
        b = torch.from_numpy(np.stack([fonte[i:i + ctx + 1] for i in ini]).astype(np.int64)).to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            r.append(modelo(b[:, :-1], b[:, 1:])[1].item())
    return float(np.mean(r))

@torch.no_grad()
def copia(n=64, k=50):
    modelo.eval(); g = torch.Generator().manual_seed(SEED)
    seq = torch.randint(10, CFG["vocabulario"], (n, k), generator=g)
    x = torch.cat([seq, seq], 1).to(dispositivo)
    with torch.autocast(device_type="cuda", dtype=DTYPE):
        lg = modelo(x)[0]
    return float((lg[:, k:-1].argmax(-1) == x[:, k + 1:]).float().mean())

def medir(passo):
    m = {"passo": passo, "tokens": passo * LOTE * CTX, "perda_proprio_256": perda(PROPRIO_VAL, 256), "perda_web_1024": perda(WEB_VAL, CTX, lotes=15),
         "copia_top1": copia()}
    print(f"  MEDIDA passo {passo}: próprio(256) {m['perda_proprio_256']:.4f} | web(1024) {m['perda_web_1024']:.4f} | cópia {100*m['copia_top1']:.1f}%", flush=True)
    return m

if PASSO0 == 0:
    HIST.append(medir(0))
t0, passo = time.time(), PASSO0
for passo in range(PASSO0 + 1, PASSOS + 1):
    modelo.train()
    for g in otim.param_groups:
        g["lr"] = taxa(passo)
    x, y = lote()
    otim.zero_grad(set_to_none=True)
    soma = 0.0
    for xp, yp in zip(x.chunk(PEDACOS), y.chunk(PEDACOS)):
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            l = rodar(xp, yp)[1] / PEDACOS
        escala.scale(l).backward(); soma += l.item()
    l = torch.tensor(soma)
    escala.unscale_(otim)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    escala.step(otim); escala.update()
    if passo % 200 == 0:
        print(f"passo {passo}/{PASSOS} perda {l.item():.3f} lr {taxa(passo):.2e} | {passo*LOTE*CTX/1e9:.2f} bi de tokens | {(time.time()-t0)/max(1,passo-PASSO0):.2f}s/passo", flush=True)
    if passo % MEDIR_A_CADA == 0 or passo == PASSOS:
        HIST.append(medir(passo))
    fim_tempo = time.time() - t0 > MAX_HORAS * 3600
    if passo % SALVAR_A_CADA == 0 or passo == PASSOS or fim_tempo:
        tmp = SAIDA / "checkpoint.tmp"
        torch.save({"modelo": modelo.state_dict(), "otim": otim.state_dict(), "passo": passo, "historico": HIST, "config": CFG,
                    "commit": COMMIT}, tmp)
        tmp.replace(CKPT)
        print(f"  checkpoint salvo (passo {passo})", flush=True)
    if fim_tempo:
        print("Limite de tempo da sessão: rode 'Executar tudo' de novo para continuar.", flush=True)
        break
TERMINOU = passo >= PASSOS


In [ ]:
# Célula 5 — exportar a base crescida (só no fim dos passos) e o resumo pequeno
resumo = {"protocolo": "fase7a_crescer_v7c_e_ler", "criado_em": datetime.now().isoformat(), "commit": COMMIT, "gpu": GPU,
          "base_sha256": BASE_SHA, "config": CFG, "parametros": sum(p.numel() for p in modelo.parameters()),
          "passo": passo, "passos": PASSOS, "terminou": TERMINOU, "tokens_lidos": passo * LOTE * CTX,
          "corpus": {"web_tokens": int(sum(len(w) for w in WEB)), "proprio_tokens": int(len(PROPRIO)), "frac_web": FRAC_WEB,
                     "fonte_web": man["fonte"]},
          "historico": HIST, "ativo_no_chat": False}
if TERMINOU:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    DEST = SAIDA / f"base_{stamp}"
    DEST.mkdir(parents=True)
    torch.save({"versao": lp.VERSAO, "modelo": {k: v.detach().cpu() for k, v in modelo.state_dict().items()}, "config": CFG,
                "passo": passo, "execucao": {"tokenizer_sha256": sha256(BASE.parent / "tokenizer.json"), "pesos_externos": False,
                                              "crescido_de": BASE_SHA, "protocolo": "fase7a", "crescimento": "scripts/crescer_46m.py, 768 × 40, ruído 1e-4", "commit": COMMIT}}, DEST / "pesos.pt")
    shutil.copyfile(BASE.parent / "tokenizer.json", DEST / "tokenizer.json")
    resumo.update(pasta=str(DEST), pesos_sha256=sha256(DEST / "pesos.pt"))
    print("BASE CRESCIDA SALVA EM", DEST, "| sha", resumo["pesos_sha256"], flush=True)
(SAIDA / "resumo_fase7a.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=1, default=float))
print(json.dumps({k: resumo[k] for k in ("passo", "passos", "terminou", "tokens_lidos")}, indent=1), flush=True)
print("Avise o assistente: ele lê o resumo_fase7a.json." if TERMINOU else "Ainda não terminou: rode 'Executar tudo' de novo para continuar.", flush=True)
